# 16b · GSE232381 · bulk_RNA_seq · do the paper's genes differ between active and inactive nephritis?

Reads the NCBI raw counts, NCBI gene_info and `metadata.rds`.

**Claim C2** (Chen YC et al. *Heliyon* 2024;10:e32303, Table 3): 28 molecules differ between active and
inactive lupus nephritis, found with DESeq2 or edgeR (Methods) on 9 active and 9 inactive patients.
Table 3 gives a log2 fold change for each.

**Which genes can be tested.** Two rows of Table 3 are not single genes ("26s Proteasome complex",
"PI3K (complex)") and are not tested. Where the table names a group and a gene (Jnk | MAPK8, MTORC2 |
RICTOR, MTORC2 | MLST8, Notch | NOTCH4, RAS | RRAS, HSP | DNAJB14), the gene is tested. IGLL1/IGLL5 is
tested as both genes. That leaves 27 genes.

**Samples.** The 16 samples with NCBI counts (10 active, 6 inactive; notebook 11). This is not the
paper's set of 9 and 9, so agreement is not expected to be exact.

**Two tests, side by side** (active against inactive, all genes, then the paper's genes extracted):
- **DESeq2** (Love MI, Huber W, Anders S. *Genome Biol* 2014;15:550, doi:10.1186/s13059-014-0550-8),
  the method the paper names; default settings, Wald test, Benjamini-Hochberg adjusted p.
- **limma-voom** (Law CW, Chen Y, Shi W, Smyth GK. *Genome Biol* 2014;15:R29, doi:10.1186/gb-2014-15-2-r29),
  after `filterByExpr` and TMM normalisation (Robinson MD, Oshlack A. *Genome Biol* 2010;11:R25).

**Tests of agreement with the paper.**
1. Direction: the number of genes whose log2 fold change has the paper's sign.
2. Size: Spearman correlation between our log2 fold changes and the paper's.
3. Significance: the number of the paper's genes with adjusted p < 0.05 here.

## Read the raw counts

In [1]:
source("../src/paths.R")
suppressMessages({library(DESeq2); library(edgeR); library(limma)})
meta <- readRDS(art("GSE232381", "metadata.rds"))
cnt  <- read.delim(raw("GSE232381", "GSE232381_raw_counts_GRCh38.p13_NCBI.tsv.gz"), check.names = FALSE)
gi   <- read.delim(file.path(NCBI, "Homo_sapiens.gene_info.gz"), quote = "", colClasses = "character")
sym  <- gi$Symbol[match(cnt$GeneID, gi$GeneID)]
counts <- as.matrix(cnt[!is.na(sym), meta$sample]); counts <- rowsum(counts, sym[!is.na(sym)])
meta$activity <- factor(ifelse(meta$ln_active == 1, "active", "inactive"), levels = c("inactive", "active"))
c(genes = nrow(counts), samples = ncol(counts)); table(meta$activity)

genes samples 
  37662      16


inactive   active 
       6       10 

**Explanation**
- `library(DESeq2)`, `library(edgeR)` and `library(limma)` provide the two tests (citations in the
  introduction). Both need raw counts, not the normalized values of notebook 12.
- The counts are read and named by symbol, and rows sharing a symbol are summed, as in notebook 12. No
  filter is applied here: each method applies its own.
- `activity` is a factor with inactive first, so the comparison reads active against inactive.

## The paper's genes and log2 fold changes

In [2]:
paper <- data.frame(
  gene = c("CD83", "E2F4", "HBEGF", "HCST", "HLA-G", "DNAJB14", "IGLL1", "IGLL5", "IRAK1", "ITGAV", "MAPK8",
           "MPG", "RICTOR", "MLST8", "NFKBIZ", "NOTCH4", "NR3C1", "PIK3CA", "PKM", "RRAS", "SMAD4", "SP3",
           "TANK", "TBK1", "TGFBR2", "TNFSF12", "ZC3H12A"),
  paper_log2FC = c(-3.14774, -0.49037, -3.57431, -0.67715, -1.55851, 1.327, -3.06243, -3.06243, -0.44746, 1.105709,
                   -2.338, -0.70755, 1.134, -0.630, -4.29614, -1.12, 0.698912, 0.865196, -0.60801, -0.941, 0.722033,
                   0.698179, 0.491583, 0.558364, 0.599042, -0.44374, -2.50818))
c(paper_genes = nrow(paper), counted_here = sum(paper$gene %in% rownames(counts)))

paper_genes counted_here 
          27           27

**Explanation**
- `paper` holds the 27 testable genes of Table 3 and the log2 fold change the paper reports for each. The
  numbers are copied from Table 3.
- `counted_here` counts how many of them are in the counts table.

**DESeq2**, design `~ activity`, active against inactive.

In [3]:
dds <- DESeqDataSetFromMatrix(counts, colData = meta, design = ~ activity)
dds <- suppressMessages(DESeq(dds, quiet = TRUE))
r_deseq <- as.data.frame(results(dds, contrast = c("activity", "active", "inactive")))
c(genes_tested = sum(!is.na(r_deseq$padj)), padj_below_0.05 = sum(r_deseq$padj < 0.05, na.rm = TRUE))

genes_tested padj_below_0.05 
          12094               1

**Explanation**
- `DESeqDataSetFromMatrix()` builds DESeq2's object from the counts, the sample table and the design
  `~ activity`.
- `DESeq()` estimates size factors and dispersions and fits the model; `quiet = TRUE` hides its messages.
- `results(dds, contrast = c("activity", "active", "inactive"))` gives, for each gene, the log2 fold change of
  active against inactive, its p value and its Benjamini-Hochberg adjusted p value (`padj`).
- DESeq2 sets `padj` to NA for genes it does not test (very low counts or outliers); `!is.na()` counts the
  tested ones.

**Result.** Genome-wide, DESeq2 finds one gene with adjusted p < 0.05 among 12,094 tested.

**limma-voom**, same design.

In [4]:
design <- model.matrix(~ activity, meta)
dge <- DGEList(counts); keep <- filterByExpr(dge, design)
dge <- calcNormFactors(dge[keep, , keep.lib.sizes = FALSE])
fit <- eBayes(lmFit(voom(dge, design), design))
r_voom <- topTable(fit, coef = "activityactive", number = Inf, sort.by = "none")
c(genes_tested = nrow(r_voom), adj_p_below_0.05 = sum(r_voom$adj.P.Val < 0.05))

genes_tested adj_p_below_0.05 
           14692                0

**Explanation**
- `model.matrix(~ activity, meta)` builds the design matrix: an intercept and a column that is 1 for active
  samples.
- `DGEList(counts)` builds edgeR's object; `filterByExpr()` keeps genes with enough counts for this design;
  `calcNormFactors()` computes TMM normalization factors.
- `voom()` turns counts into log2 counts per million with a precision weight for each value; `lmFit()` fits
  the linear model; `eBayes()` moderates the variances.
- `topTable(fit, coef = "activityactive", number = Inf, sort.by = "none")` returns the result for the
  activity column for all genes, in their original order.

**Result.** limma-voom finds no gene with adjusted p < 0.05 among 14,692 tested.

**The paper's genes**, our log2 fold change (active against inactive), p and adjusted p from each method.

In [5]:
cmp <- data.frame(paper,
  DESeq2_log2FC = r_deseq[paper$gene, "log2FoldChange"], DESeq2_p = r_deseq[paper$gene, "pvalue"], DESeq2_padj = r_deseq[paper$gene, "padj"],
  voom_log2FC = r_voom[paper$gene, "logFC"], voom_p = r_voom[paper$gene, "P.Value"], voom_padj = r_voom[paper$gene, "adj.P.Val"])
format(cmp, digits = 3)

,gene,paper_log2FC,DESeq2_log2FC,DESeq2_p,DESeq2_padj,voom_log2FC,voom_p,voom_padj
,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>
1,CD83,-3.148,0.80107,0.08898,0.393,0.552,0.28592,0.672
2,E2F4,-0.490,-0.36594,0.09172,0.395,-0.336,0.11676,0.512
3,HBEGF,-3.574,0.50269,0.44503,NA,0.254,0.68681,0.892
4,HCST,-0.677,-0.66901,0.00375,0.164,-0.601,0.01252,0.359
5,HLA-G,-1.559,0.10649,0.86120,0.948,0.528,0.38877,0.743
6,DNAJB14,1.327,0.82272,0.05291,0.323,0.742,0.11413,0.510
7,IGLL1,-3.062,-2.42491,0.03866,NA,NA,NA,NA
8,IGLL5,-3.062,-2.12712,NA,NA,-0.631,0.69715,0.895
9,IRAK1,-0.447,-0.30418,0.06417,0.349,-0.303,0.08535,0.471


**Explanation**
- One row per paper gene: the paper's log2 fold change next to ours from each method, with p and adjusted p.
  `r_deseq[paper$gene, "log2FoldChange"]` looks up each gene by name.

**Agreement with the paper**, over the genes each method tested.

In [6]:
agree <- function(ours, padj) {
  ok <- !is.na(ours)
  c(genes = sum(ok), same_direction = sum(sign(ours[ok]) == sign(cmp$paper_log2FC[ok])),
    spearman_rho = round(cor(ours[ok], cmp$paper_log2FC[ok], method = "spearman"), 3),
    spearman_p = signif(cor.test(ours[ok], cmp$paper_log2FC[ok], method = "spearman", exact = FALSE)$p.value, 3),
    adjusted_p_below_0.05 = sum(padj[ok] < 0.05, na.rm = TRUE))
}
rbind(DESeq2 = agree(cmp$DESeq2_log2FC, cmp$DESeq2_padj), limma_voom = agree(cmp$voom_log2FC, cmp$voom_padj))

,genes,same_direction,spearman_rho,spearman_p,adjusted_p_below_0.05
DESeq2,27,23,0.563,0.00222,0
limma_voom,26,21,0.606,0.00103,0


**Explanation**
- `agree(ours, padj)` compares one method with the paper over the genes it tested:
  - `same_direction`: genes whose log2 fold change has the paper's sign;
  - `spearman_rho` and `spearman_p`: Spearman rank correlation of our fold changes with the paper's, and its
    p value. `exact = FALSE` uses the normal approximation, as ties are possible;
  - `adjusted_p_below_0.05`: the paper's genes significant here.
- `rbind()` puts the rows of the two methods together.

**Result: claim C2 partly reproduced.**
- **Direction.** 23 of 27 genes (DESeq2) and 21 of 26 (limma-voom) change in the paper's direction, and
  our log2 fold changes correlate with the paper's (Spearman rho = 0.56, p = 0.002; rho = 0.61, p = 0.001).
- **The paper's five largest effects are not reproduced**: NFKBIZ (paper -4.30; here -0.00 and +0.21),
  HBEGF (-3.57; +0.50 and +0.25), CD83 (-3.15; +0.80 and +0.55), ZC3H12A (-2.51; -0.25 and -0.34) and
  MAPK8 (-2.34; +0.31 and +0.38). The agreement comes from the genes with modest effects (for example
  RICTOR, NR3C1, SP3, MPG, MLST8, HCST).
- **Significance.** None of the paper's genes has adjusted p < 0.05 here with either method (smallest
  0.14 with DESeq2, 0.36 with limma-voom). These are 16 samples, not the paper's 18, and 10 against 6
  rather than 9 against 9.